# SQL with Python
source: https://youtu.be/eDXX5evRgQw?si=OXtRQfhN-Zgui-3q

#### importing libraries

In [2]:
import pyodbc
import pandas

#### creating data in python
I create this list of lists that I will want to insert into a table in SQL Server later.

In [14]:
# my price data
price_data = [[2.00, 3.00, 1.00, 2.40, 100.00, '1/2/2019'],
              [3.00, 3.00, 5.00, 9.40, 300.00, '2/1/2020'],
              [4.00, 2.00, 1.00, 2.40, 200.00, '3/1/2021']]

#### Creating the connect string
In order to connect to the database, I need to create this connect string.

In [ ]:
# connection string
connection_string = 'DRIVER={SQL Server};SERVER=<srv>'
#connection_string = 'DRIVER={SQL Server};SERVER=<srv>,<port>'
#connection_string = 'DRIVER={SQL Server};SERVER=<srv>,<port>;DATABASE=<db_name>'
#connection_string = 'DRIVER={SQL Server};SERVER=<srv>,<port>;DATABASE=<db_name>;Trusted_Connection=yes'

In [ ]:
# to find you driver you can use:
for driver in pyodbc.drivers():
    print(driver)

SQL Server
iAnywhere Solutions 16 - Oracle
Sybase IQ
Teradata Database ODBC Driver 20.00
Teradata 8.0 DB2 Wire Protocol
Teradata 8.0 Oracle Wire Protocol
Teradata 8.0 SQL Server Wire Protocol
Teradata 8.0 MySQL Wire Protocol
Teradata 8.0 PostgreSQL Wire Protocol
Teradata Database ODBC Driver 16.20
SQL Server Native Client 11.0
ODBC Driver 13 for SQL Server
ODBC Driver 17 for SQL Server


#### Queries order
1. create the <b>query string</b>
1. create cursor using your connect string
1. use the cursor to execute your query from your <b>query string</b>
1. (if needed) commit the changes
    1. commiting the changes is usually required with DDL/DML/..., but not with DQL
1. close the connection by closing the cursor

#### CREATE query

In [78]:
# define the query
create_query = '''CREATE TABLE td_price_data(close_price float, 
                                            high float, 
                                            low float, 
                                            open_price float, 
                                            volume float, 
                                            day_volume date)'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(create_query)

# commit the changes
cnxn.commit()

# close the cursor
cursor.close()

# close the connection
cnxn.close()

#### SELECT query (on an empty table)

In [69]:
# define the query
query_select1 = '''SELECT * FROM [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(query_select1)

# commit the changes
#cnxn.commit()

# grab all the rows in our database table
for row in cursor:
    print(row)

# close the cursor
cursor.close()

# close the connection
cnxn.close()

#### INSERT query
I am printing our the values only so that it's clear what is happening in the loop.

In [79]:
# define the query
query_insert = '''INSERT INTO td_price_data (close_price, high, low, open_price, volume, day_volume)
                  VALUES (?, ?, ?, ?, ?, ?);''' # the "?" signs are like placeholders (sometimes "%" is used -> this does not work here)

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
for row in price_data:
    
    # define the values to insert
    values = (row[0], row[1], row[2], row[3], row[4], row[5])
    print(values)

    # insert the data into the database table
    cursor.execute(query_insert, values)



# commit the changes
cnxn.commit()

# close the cursor
cursor.close()

# close the connection
cnxn.close()

(2.0, 3.0, 1.0, 2.4, 100.0, '1/2/2019')
(3.0, 3.0, 5.0, 9.4, 300.0, '2/1/2020')
(4.0, 2.0, 1.0, 2.4, 200.0, '3/1/2021')


#### SELECT query on the table with values

In [71]:
# define the query
query_select = '''SELECT * FROM [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(query_select1)

# commit the changes
#cnxn.commit()

# grab all the rows in our database table
for row in cursor:
    print(row)

# close the cursor
cursor.close()

# close the connection
cnxn.close()

(2.0, 3.0, 1.0, 2.4, 100.0, '2019-01-02')
(3.0, 3.0, 5.0, 9.4, 300.0, '2020-02-01')
(4.0, 2.0, 1.0, 2.4, 200.0, '2021-03-01')


#### Saving the data into a list

In [73]:
# define the query
query_select = '''SELECT * FROM [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(query_select)

# fetch the result
results = cursor.fetchall()  # Fetch results here

# close the cursor
cursor.close()

# close the connection
cnxn.close()

# explore the results
print(results)
type(results)

[(2.0, 3.0, 1.0, 2.4, 100.0, '2019-01-02'), (3.0, 3.0, 5.0, 9.4, 300.0, '2020-02-01'), (4.0, 2.0, 1.0, 2.4, 200.0, '2021-03-01')]


list

#### Saving data into a DataFrame

In [74]:
# Define the SQL query
query_select = '''SELECT * FROM [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# Execute the query and store results in a DataFrame
df = pd.read_sql_query(query_select, cnxn)

# Optionally, close the connection
cnxn.close()

# Now `df` contains the results from the query
print(df.head())  # Display the first few rows


   close_price  high  low  open_price  volume  day_volume
0          2.0   3.0  1.0         2.4   100.0  2019-01-02
1          3.0   3.0  5.0         9.4   300.0  2020-02-01
2          4.0   2.0  1.0         2.4   200.0  2021-03-01


#### DROP query

In [76]:
# define the query
query_drop = '''DROP TABLE [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(query_drop)

# commit the changes
cnxn.commit()

# close the cursor
cursor.close()

# close the connection
cnxn.close()


#### SELECT query (so we see the table does not exist anymore)

In [77]:
# define the query
query_select = '''SELECT * FROM [GLOW001\JF16783].td_price_data'''

# connect to the database
cnxn = pyodbc.connect(connection_string)

# create the connection cursor
cursor = cnxn.cursor()

# execute the query
cursor.execute(query_select)

# commit the changes
#cnxn.commit()

# grab all the rows in our database table
for row in cursor:
    print(row)

# close the cursor
cursor.close()

# close the connection
cnxn.close()

ProgrammingError: ('42S02', "[42S02] [Microsoft][ODBC SQL Server Driver][SQL Server]Invalid object name 'GLOW001\\JF16783.td_price_data'. (208) (SQLExecDirectW)")